In [1]:
# Data handling
import ast
import pandas as pd
import numpy as np

# Machine learning: data splitting and evaluation
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

# Text feature extraction
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer
)

# Machine learning models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import LinearSVC

# Model performance metrics
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Visualization
import matplotlib.pyplot as plt

### Load in the data and define X, y, train and test sets

In [2]:
df = pd.read_csv('./data/emotions_clean.csv')

X = df['clean_tokens']
y = df['emotion']

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=df["emotion"]
)

### Perform Linear SVM without feature selection as a baseline

In [3]:
# Create the pipeline
svm_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("model", LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    ))
])

# fit and predict
svm_pipe.fit(X_train, y_train)

y_pred_svm = svm_pipe.predict(X_test)

# Evaluate performance
print(classification_report(
    y_test, y_pred_svm, digits=3
))

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_svm))
print("Macro F1:", f1_score(y_test, y_pred_svm, average='macro'))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_svm))

              precision    recall  f1-score   support

       anger      0.915     0.873     0.893       432
        fear      0.897     0.879     0.888       387
         joy      0.892     0.928     0.910      1072
        love      0.815     0.743     0.778       261
     sadness      0.919     0.939     0.929       933
    surprise      0.843     0.748     0.793       115

    accuracy                          0.896      3200
   macro avg      0.880     0.852     0.865      3200
weighted avg      0.895     0.896     0.895      3200

Accuracy: 0.89625
Balanced accuracy: 0.8515729428462738
Macro F1: 0.864955297395884
Confusion matrix:
[[377   8  14   6  27   0]
 [  8 340  16   2  13   8]
 [  9   2 995  34  27   5]
 [  2   1  55 194   9   0]
 [ 15   8  29   2 876   3]
 [  1  20   7   0   1  86]]


### Perform a grid search of SVM hyperparameters without feature selection


In [4]:
# Define SVM pipeline
svm_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("model", LinearSVC(
        max_iter=5000,
        random_state=42
    ))
])

# Define hyperparameter grid
param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 3, 5],
    'tfidf__max_df': [0.90, 0.95, 1.0]
}

In [5]:
# Run GridSearchCV and examine results
svm_grid = GridSearchCV(
    svm_pipe,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

svm_grid.fit(X_train, y_train)

print("Best parameters:", svm_grid.best_params_)
print("Best CV Macro F1:", svm_grid.best_score_)


Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'model__C': 1, 'tfidf__max_df': 0.95, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
Best CV Macro F1: 0.8502862555536252


### Create a pipeline that TFIDF encodes the tokens and selects the 5000 most informative features using linear SVM

In [6]:
# Create SVM pipeline with Chi-squared feature selection
svm_fs_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("feature_selection", SelectKBest(
        score_func=chi2
    )),
    ("model", LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    ))
])

# define parameter grid for selected features
param_grid_fs = {
    'feature_selection__k': [500, 1000, 2500, 5000, 10000],
    'model__C': [0.1, 1, 10]
}

In [7]:
# Run grid search and examine results
svm_fs_grid = GridSearchCV(
    svm_fs_pipe,
    param_grid_fs,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

svm_fs_grid.fit(X_train, y_train)

print("Best parameters:", svm_fs_grid.best_params_)
print("Best CV Macro F1:", svm_fs_grid.best_score_)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


/home/mark/OMDS/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=10000 is greater than n_features=8952. All the features will be returned.
  warnings.warn(
/home/mark/OMDS/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=10000 is greater than n_features=9018. All the features will be returned.
  warnings.warn(
/home/mark/OMDS/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=10000 is greater than n_features=9053. All the features will be returned.
  warnings.warn(
/home/mark/OMDS/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=10000 is greater than n_features=9040. All the features will be returned.
  warnings.warn(
/home/mark/OMDS/.venv/lib/python3.12/site-packages/sklearn/feature_selection/_univariate_selection.py:782: UserWarning: k=10000 is greater than n_fe

Best parameters: {'feature_selection__k': 2500, 'model__C': 1}
Best CV Macro F1: 0.8593813289894177


### Now that best hyperparameters and number of features have been selected, run SVM, TFIDF, and SelectKBest() with best parameters

In [8]:
# Define pipeline with best parameters
svm_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("feature_selection", SelectKBest(
        score_func=chi2,
        k=2500
    )),
    ("model", LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    ))
])

In [9]:
# Fit model, make predictions, and evaluate
svm_pipe.fit(X_train, y_train)

y_pred_svm = svm_pipe.predict(X_test)

from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    confusion_matrix,
    accuracy_score,
    f1_score
)

print(classification_report(
    y_test, y_pred_svm, digits=3
))

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_svm))
print("Macro F1:", f1_score(y_test, y_pred_svm, average='macro'))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_svm))


              precision    recall  f1-score   support

       anger      0.927     0.877     0.901       432
        fear      0.895     0.907     0.901       387
         joy      0.878     0.941     0.909      1072
        love      0.841     0.709     0.769       261
     sadness      0.940     0.941     0.941       933
    surprise      0.865     0.722     0.787       115

    accuracy                          0.902      3200
   macro avg      0.891     0.850     0.868      3200
weighted avg      0.901     0.902     0.900      3200

Accuracy: 0.9015625
Balanced accuracy: 0.8495207780649334
Macro F1: 0.8679283303272435
Confusion matrix:
[[ 379    5   25    4   19    0]
 [   8  351    9    1   14    4]
 [  11    2 1009   29   15    6]
 [   1    1   66  185    8    0]
 [  10    9   32    1  878    3]
 [   0   24    8    0    0   83]]


In [10]:
# Examine feature selection
tfidf = svm_pipe.named_steps['tfidf']
selector = svm_pipe.named_steps['feature_selection']

print("Original TF-IDF features:", len(tfidf.get_feature_names_out()))
print("Selected features:", selector.get_support().sum())

Original TF-IDF features: 10984
Selected features: 2500


In [11]:
# Examine highest scoring features
feature_scores = pd.DataFrame({
    'feature': tfidf.get_feature_names_out(),
    'score': selector.scores_
})

print(feature_scores.sort_values(
    'score', ascending=False
).head(30))

                feature       score
245              amazed  444.187530
4871          impressed  375.305114
1728            curious  368.407540
2789        feel amazed  363.379043
9274          surprised  300.839025
2895       feel curious  277.126517
4015              funny  275.880270
3021         feel funny  259.949290
8540            shocked  238.257240
3406     feel surprised  232.794820
10668             weird  202.901367
7028        overwhelmed  200.595582
1850               daze  194.774063
9328        sympathetic  186.402182
3239   feel overwhelmed  179.095925
3104     feel impressed  172.657884
4687              horny  168.283919
3500         feel weird  164.268897
9049            strange  160.098706
6681            naughty  159.265446
6843          nostalgic  157.424435
6251              loyal  153.137602
3347       feel shocked  146.865770
1927           delicate  136.826126
9130            stunned  135.122021
3409   feel sympathetic  134.826075
9237         supportive  134

In [13]:
tfidf_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("classifier", MultinomialNB(alpha=1.0))
])

tfidf_model.fit(X_train, y_train)
y_pred = tfidf_model.predict(X_test)

print(classification_report(
    y_test, y_pred, digits=3
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

       anger      0.965     0.581     0.725       432
        fear      0.898     0.594     0.715       387
         joy      0.689     0.968     0.805      1072
        love      0.980     0.188     0.315       261
     sadness      0.776     0.930     0.846       933
    surprise      1.000     0.078     0.145       115

    accuracy                          0.764      3200
   macro avg      0.885     0.557     0.592      3200
weighted avg      0.812     0.764     0.732      3200

Accuracy: 0.7640625
Balanced accuracy: 0.5566583234624715
Macro F1: 0.5920805802224651
Confusion matrix:
[[ 251    2   79    0  100    0]
 [   5  230   91    0   61    0]
 [   2    0 1038    1   31    0]
 [   1    0  182   49   29    0]
 [   1    1   63    0  868    0]
 [   0   23   54    0   29    9]]
